In [1]:
import cartopy.crs as ccrs
import xarray as xr
import pandas as pd
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib import pyplot as plt
import glob
import matplotlib
%matplotlib inline

In [2]:
#define focusing area
[min_lon_plot, max_lon_plot, min_lat_plot, max_lat_plot]=[100.,150.,12.,48.]
[min_lon_tw, max_lon_tw, min_lat_tw, max_lat_tw]=[118,126,22,26]
[min_lon_ish, max_lon_ish, min_lat_ish, max_lat_ish]=[122,126,22,26]
slp_levels=np.linspace(980,1040,16)
slp_lws=[2 if (pp-980)%20==0 else 1 for pp in slp_levels]
prec_threshold=0.1
ishigaki={'lat':24.33,'lon':124.16}
m='TaiESM1'

In [3]:
ds_yr=xr.open_dataset('../data/regridded_%s_2015.nc'%m)
ds_yr

<xarray.Dataset>
Dimensions:        (time: 365, lat: 42, lon: 41)
Coordinates:
  * time           (time) datetime64[ns] 2015-01-01 2015-01-02 ... 2015-12-31
  * lat            (lat) float64 10.84 11.78 12.72 13.66 ... 47.59 48.53 49.48
  * lon            (lon) float64 100.0 101.2 102.5 103.8 ... 147.5 148.8 150.0
    level          float64 ...
Data variables:
    precipitation  (time, lat, lon) float32 ...
    u              (time, lat, lon) float32 ...
    v              (time, lat, lon) float32 ...
    msl            (time, lat, lon) float32 ...
    LTS            (time, lat, lon) float32 ...
Attributes:
    DimensionNames:    time,lon,lat
    Units:             mm/hr
    units:             mm/hr
    CodeMissingValue:  -9999.9
    LongName:          \nComplete merged microwave-infrared (gauge-adjusted)\...
    regrid_method:     conservative

In [4]:
def uv2ws(u, v):
    func = lambda x, y: np.sqrt(x**2 + y**2)
    return xr.apply_ufunc(func, u, v,dask="allowed")

def uv2wd(u, v):
    func = lambda x,y: np.where(u==0,np.where(v>0,180.0,0.0),180.0 + np.arctan2(u, v) * 180.0 / np.pi)
    return xr.apply_ufunc(func, u, v,dask="allowed")

def calc_wd_cor(wd_array):
    wd_min=wd_array.min()
    #print(wd_min)
    wdList=np.array([dd-wd_min if abs(dd-wd_min)<180.0 else wd_min-(dd-360.0) for dd in wd_array.ravel()])
    result=np.percentile(wdList,75)
    return result
def uv2deg_ish(ser):
    if ser.ish_u==0:
        return 180.0 if ser.ish_v>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.ish_v/ser.ish_u)/np.pi*180
        return tmpdir if ser.ish_u>0 else tmpdir-180
    
def uv2deg_mean(ser):
    if ser.meanU==0:
        return 180.0 if ser.meanV>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.meanV/ser.meanU)/np.pi*180
        return tmpdir if ser.meanU>0 else tmpdir-180

In [8]:
df_list=[]
#yr_start=2015
#yr_end=2020
yr_start=2005
yr_end=2014
for yr in range(yr_start,yr_end+1):
    ds_yr=xr.open_dataset('../data/regridded_%s_%04d.nc'%(m,yr))
    tw_ds=ds_yr.sel(lat=slice(min_lat_tw,max_lat_tw),lon=slice(min_lon_tw,max_lon_tw))
    tw_ds['prRatio']=tw_ds.precipitation.where(tw_ds.precipitation>=prec_threshold).count(dim=['lon','lat'])/30.0
    df_tw=pd.DataFrame({'date':tw_ds.time.values,'prRatio':tw_ds.prRatio.values})
    
    ish_ds=ds_yr.sel(lat=slice(min_lat_ish,max_lat_ish),lon=slice(min_lon_ish,max_lon_ish))
    ish_ds['ws']=uv2ws(ish_ds.u, ish_ds.v)
    ish_ds['wd']=uv2wd(ish_ds.u, ish_ds.v)
    #ish_ds['wd_p90']=ish_ds.wd.reduce(np.percentile, q=90,dim=['lat','lon'])
    #ish_ds['wd_p10']=ish_ds.wd.reduce(np.percentile, q=10,dim=['lat','lon'])
    #ish_ds['wdCor_raw']=ish_ds.wd_p90-ish_ds.wd_p10
    #ish_ds['wdCor']=ish_ds.wdCor_raw.where(ish_ds.wdCor_raw<180,360-ish_ds.wdCor_raw)
    ish_ds['wdCor']=xr.apply_ufunc(calc_wd_cor,ish_ds.wd,input_core_dims=[['lat','lon']],
                                   output_core_dims=[[]],vectorize=True)
    ish_ds['ws_mean']=ish_ds.ws.reduce(np.mean,dim=['lat','lon'])
    ish_ds['maxWS']=ish_ds.ws.reduce(np.max,dim=['lat','lon'])
    ish_ds['stdU']=ish_ds.u.reduce(np.std,dim=['lat','lon'])
    ish_ds['stdV']=ish_ds.v.reduce(np.std,dim=['lat','lon'])
    ish_ds['std_wind_component']=0.5*(ish_ds.stdU+ish_ds.stdV)
    ish_ds['u_mean']=ish_ds.u.reduce(np.mean,dim=['lat','lon'])
    ish_ds['v_mean']=ish_ds.v.reduce(np.mean,dim=['lat','lon'])  
    ish_ds['LTS_mean']=ish_ds.LTS.reduce(np.mean,dim=['lat','lon'])  
    idx=ish_ds.wdCor.idxmax(dim='time')
    dateStr=ish_ds.wdCor.idxmax(dim='time').dt.strftime('%Y-%m-%d').item()
    print(dateStr,pd.to_datetime(dateStr).dayofyear,ish_ds.wdCor.max().item())
    #save to dataframe
    yyyymmdd=ish_ds.time.dt.strftime('%Y%m%d')
    u_ish=ish_ds.u.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
    v_ish=ish_ds.v.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
    LTS_ish=ish_ds.LTS.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
    wdCor=ish_ds.wdCor.values
    df_ish=pd.DataFrame({'date':ish_ds.time.values,'yyyymmdd':yyyymmdd, 
                         'meanU':ish_ds.u_mean.values,'meanV':ish_ds.v_mean.values,'meanLTS':ish_ds.LTS_mean.values,
                         'wdCor':wdCor,'meanWS':ish_ds.ws_mean.values,'maxWS':ish_ds.maxWS.values,'std_uv':ish_ds.std_wind_component.values,
                         'ish_u':u_ish,'ish_v':v_ish,'ish_LTS':LTS_ish,})
    df_ish['meanWD']=df_ish.apply(uv2deg_mean,axis=1)
    df_ish['ish_wd']=df_ish.apply(uv2deg_ish,axis=1)
    df_ish['ish_ws']=df_ish.apply(lambda ser: np.sqrt(ser.ish_u*ser.ish_u+ser.ish_v*ser.ish_v),axis=1)
    df_ish['year']=df_ish.apply(lambda ser: ser.date.year,axis=1)
    df_ish['month']=df_ish.apply(lambda ser: ser.date.month,axis=1)
    
    final=pd.merge(df_ish,df_tw,on='date')
    #final.to_csv('../data/flow_regime.wind1000.%s.%04d.csv'%('reanalysis',yr),index=False)    
    df_list.append(final)

2005-03-22 81 177.81678771972656
2006-06-06 157 176.89409637451172
2007-03-15 74 175.8924331665039
2008-01-28 28 161.53148651123047
2009-03-05 64 176.04083251953125
2010-04-12 102 175.74169158935547
2011-04-16 106 176.90215301513672
2012-05-15 136 175.7652816772461
2013-06-02 153 176.49853515625
2014-05-18 138 173.1207504272461


In [9]:
all_yr=pd.concat(df_list)
all_yr.to_csv('../data/reanalysis.%s_grid.%d-%d.indices.csv'%(m,yr_start,yr_end), index=False)
all_yr

,date,yyyymmdd,meanU,meanV,meanLTS,wdCor,meanWS,maxWS,std_uv,ish_u,ish_v,ish_LTS,meanWD,ish_wd,ish_ws,year,month,prRatio
0,2005-01-01,20050101,-2.795265,-10.001824,19.122082,18.657623,10.524763,12.376671,1.183258,-1.551010,-9.994319,19.174556,15.614378,8.821317,10.113953,2005,1,0.400000
1,2005-01-02,20050102,-4.528355,-1.896448,14.000731,22.449280,5.024467,8.115359,1.498908,-3.959663,-1.809919,13.859595,67.276361,65.435372,4.353704,2005,1,0.100000
2,2005-01-03,20050103,-3.519047,-1.003856,13.143312,58.845276,4.211099,8.174035,2.078828,-3.197777,0.167876,12.790764,74.078474,93.005141,3.202181,2005,1,0.133333
3,2005-01-04,20050104,-5.485624,-9.531304,15.437488,14.388229,11.055293,12.272929,1.092143,-5.226576,-9.930520,14.897537,29.922044,27.758477,11.221957,2005,1,0.466667
4,2005-01-05,20050105,-5.793207,-3.189214,16.716722,24.384983,6.726004,8.874956,1.244268,-6.266853,-2.605205,16.412014,61.166807,67.426767,6.786791,2005,1,0.133333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2014-12-27,20141227,-6.688735,-5.318996,13.457412,38.892925,8.954333,11.059999,1.973505,-6.935767,-4.720041,13.095519,51.507684,55.763291,8.389496,2014,12,0.400000
361,2014-12-28,20141228,-2.099710,-9.231874,13.589228,20.133720,9.564683,14.325666,2.231055,-2.325751,-10.499607,13.602302,12.813451,12.489824,10.754109,2014,12,0.300000
362,2014-12-29,20141229,-3.412793,-9.026219,16.414547,19.782982,9.742940,11.604609,1.365825,-2.756412,-9.137346,15.796064,20.711491,16.786696,9.544050,2014,12,0.000000
363,2014-12-30,20141230,-4.307925,-3.524414,16.557430,8.304802,5.628355,8.332024,1.323765,-3.887229,-3.403419,15.676499,50.712598,48.796608,5.166605,2014,12,0.000000
